In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

my_schema = StructType([
    StructField("call_id", StringType(), True),
    StructField("caller", StringType(), True),
    StructField("callee", StringType(), True),
    StructField("start_ts", TimestampType(), True),
    StructField("duration_s", IntegerType(), True),
    StructField("status", StringType(), True)
])

df = (
    spark.read
    .schema(my_schema)
    .option("header", True)
    .csv("/Volumes/workspace/default/raw/cdrs.csv")
)


result = (
    df.groupBy("caller")
      .agg(
          F.sum(
              F.when(
                  F.col("status") == "ANSWERED",
                  F.col("duration_s")
              ).otherwise(0)
          ).alias("answered_duration_s"),

          F.avg(
              F.when(
                  F.col("status").isin("FAILED"),
                  1
              ).otherwise(0)
          ).alias("failure_rate"),

          F.count("*").alias("call_count")
      )
      .orderBy(F.desc("answered_duration_s"))
      .limit(5)
)

display(result)

df = spark.read.schema(my_schema).option("header", True).csv(
    "/Volumes/workspace/default/raw/cdrs.csv"
)

caller,answered_duration_s,failure_rate,call_count
+48123451014,48431,0.08771929824561403,57
+48123451007,42938,0.08888888888888889,45
+48123451021,37006,0.043478260869565216,46
+48123451020,33031,0.10638297872340426,47
+48123451002,31714,0.1875,48
